In [ ]:
from pathlib import Path

import gdt
import polars as pl

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
}

GFF3_COLUMNS = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
GENE_TYPES = {"gene", "pseudogene"}  # where the Parent chain is expected to end
UNKNOWN = "unknown"

LABEL_SYNONYMS = {
    "PLT-LHBA": "PLT-PSBZ",
    "PLT-IHBA": "PLT-PSBZ",   # typo of lhbA in the source annotation
    "PLT-YCF9": "PLT-PSBZ",
    "PLT-PBF1": "PLT-PSBN",
    "PLT-YCF6": "PLT-PETN",
    "PLT-YCF10": "PLT-CEMA",
    "PLT-YCF5": "PLT-CCSA",
    "PLT-CLPP1": "PLT-CLPP",
}

In [ ]:
def read_gff3(path: Path) -> pl.DataFrame:
    """GFF3 as a DataFrame, with ID, Parent and gene= pulled out of the attributes."""
    df = pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        new_columns=GFF3_COLUMNS,
        comment_prefix="#",
        quote_char=None,
        schema_overrides={c: pl.String for c in GFF3_COLUMNS},
    )
    attr = pl.col("attributes")
    return df.with_columns(
        pl.col("start").cast(pl.Int64),
        pl.col("end").cast(pl.Int64),
        attr.str.extract(r"(?:^|;)ID=([^;]+)").alias("ID"),
        attr.str.extract(r"(?:^|;)Parent=([^;]+)").alias("Parent"),
        attr.str.extract(r"(?:^|;)gene=([^;]+)").alias("gene"),
    )
 
 
def resolve_ancestor(
    parent: str | None, id_type: dict[str, str], id_parent: dict[str, str | None]
) -> tuple[str | None, str | None, str]:
    """Walk up the Parent chain until a gene/pseudogene.
 
    Returns (ancestor ID, ancestor type, status).
    """
    if parent is None:
        return None, None, "no_parent"
    if "," in parent:
        return parent, None, "multi_parent"
 
    current, seen = parent, set()
    while True:
        if current not in id_type:
            return current, None, "missing_parent"
        ftype = id_type[current]
        if ftype in GENE_TYPES:
            return current, ftype, "ok"
        seen.add(current)
        nxt = id_parent.get(current)
        if nxt is None or nxt in seen or "," in nxt:
            # chain ends (or loops) without reaching a gene
            return current, ftype, "no_gene_ancestor"
        current = nxt

In [ ]:
for g, subgs in groups_map.items():
    gdict_file = BASE / g / f"{g}.gdt"
    gdict = gdt.read_gdict(gdict_file)
    print(f"{g}: loaded {gdict_file.name}")
 
    def label_of(feature_id: str | None) -> str:
        if feature_id is None:
            return UNKNOWN
        entry = gdict.get(feature_id)
        if entry is None:
            return UNKNOWN
        return LABEL_SYNONYMS.get(entry.label, entry.label)
 
    for subg in subgs:
        subg_dir = BASE / g / subg
        gff3_dir = subg_dir / "gff3"
        out_dir = subg_dir / "cds_tree" / "cds"
        out_dir.mkdir(parents=True, exist_ok=True)
        tsv = pl.read_csv(subg_dir / f"{subg}.tsv", separator="\t")
        print(f"Processing {g}/{subg}: {tsv.height} ANs")
 
        frames, missing_gff = [], []
        for an in tsv["AN"]:
            gff_file = gff3_dir / f"{an}.gff3"
            if not gff_file.exists():
                missing_gff.append(an)
                continue
            df = read_gff3(gff_file)
 
            # lookups for this genome; first occurrence wins
            # (CDS pieces share an ID, but they are never parents)
            firsts = df.filter(pl.col("ID").is_not_null()).unique("ID", keep="first", maintain_order=True)
            id_type = dict(zip(firsts["ID"], firsts["type"]))
            id_parent = dict(zip(firsts["ID"], firsts["Parent"]))
 
            ancestors = [resolve_ancestor(p, id_type, id_parent) for p in df["Parent"]]
            df = df.with_columns(
                pl.lit(an).alias("AN"),
                pl.Series("gdt_label", [label_of(i) for i in df["ID"]]),
                pl.Series("gene_ancestor", [a[0] for a in ancestors], dtype=pl.String),
                pl.Series("gene_ancestor_type", [a[1] for a in ancestors], dtype=pl.String),
                pl.Series("parent_status", [a[2] for a in ancestors]),
            ).with_columns(
                pl.when(pl.col("parent_status") == "ok")
                .then(pl.col("gene_ancestor").map_elements(label_of, return_dtype=pl.String))
                .otherwise(pl.lit(UNKNOWN))
                .alias("parent_gdt_label"),
            )
            frames.append(df)
 
        if missing_gff:
            print(f"  {len(missing_gff)} ANs without a GFF3: {missing_gff[:5]}...")
 
        all_rows = pl.concat(frames).select(
            "AN", "type", "start", "end", "strand", "phase",
            "ID", "Parent", "gene", "gdt_label",
            "gene_ancestor", "gene_ancestor_type", "parent_status", "parent_gdt_label",
            "attributes",
        )
        out = out_dir / "gff3_raw.tsv"
        all_rows.write_csv(out, separator="\t")
        print(f"  {all_rows.height} rows -> {out.relative_to(subg_dir)}")
 
        # --- CDS summary
        cds = all_rows.filter(pl.col("type") == "CDS")
 
        # --- clean CDS: Parent chain reaches a gene that has a gdt label
        cds_clean = (
            cds.filter(pl.col("parent_status") == "ok")
            .filter(pl.col("parent_gdt_label") != UNKNOWN)
            .drop("phase", "type", "parent_status")
        )
        out_cds = out_dir / "gff3_cds.tsv"
        cds_clean.write_csv(out_cds, separator="\t")
        print(f"  {cds_clean.height} clean CDS rows in {cds_clean['AN'].n_unique()} ANs -> {out_cds.relative_to(subg_dir)}")
        print(f"  CDS rows: {cds.height} in {cds['AN'].n_unique()} ANs")
        print(
            cds.group_by("parent_status", "gene_ancestor_type")
            .agg(pl.len().alias("rows"), pl.col("AN").n_unique().alias("ANs"))
            .sort("rows", descending=True)
        )
        ok = cds.filter(pl.col("parent_status") == "ok")
        print(
            f"  CDS reaching a gene with a gdt label: "
            f"{ok.filter(pl.col('parent_gdt_label') != UNKNOWN).height}/{cds.height}"
        )
        unlabelled = ok.filter(pl.col("parent_gdt_label") == UNKNOWN)
        if unlabelled.height:
            print("  genes reached by CDS but missing from the gdict:")
            print(unlabelled.select("AN", "gene_ancestor", "gene").unique().sort("AN"))

In [ ]:
# print 100 rows
pl.Config.set_tbl_rows(100)

In [ ]:
# CDS that need a closer look (no Parent, broken link, pseudogene parent, unlabelled gene)
problems = cds.filter(
    (pl.col("parent_status") != "ok")
    | (pl.col("gene_ancestor_type") != "gene")
    | (pl.col("parent_gdt_label") == UNKNOWN)
)
problems.select("AN", "ID", "Parent", "gene", "parent_status", "gene_ancestor", "gene_ancestor_type", "parent_gdt_label")